In [1]:
import pandas as pd
import numpy as np

# Препроцессинг и пайплайны
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Модели
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Валидация и подбор гиперпараметров
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Зафиксируем константу для воспроизводимости
RANDOM_STATE = 42

In [5]:
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.')
common_titles = ['Mr', 'Miss', 'Mrs', 'Master']
df['Title']=df['Title'].where(df['Title'].isin(common_titles), 'Rare')

df['FamilySize']=df['SibSp'] + df['Parch'] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

features = ["Pclass", "Sex", "Age", "Fare", "Embarked", "Title", "FamilySize", "IsAlone"]
X=df[features].copy()
y=df['Survived']

print('Размер X:', X.shape)
print('Пропуски в X:')
print(X.isna().sum())
print('Типы колонок:')
print(X.dtypes)

Размер X: (891, 8)
Пропуски в X:
Pclass          0
Sex             0
Age           177
Fare            0
Embarked        2
Title           0
FamilySize      0
IsAlone         0
dtype: int64
Типы колонок:
Pclass          int64
Sex               str
Age           float64
Fare          float64
Embarked          str
Title             str
FamilySize      int64
IsAlone         int64
dtype: object


In [10]:
numeric_features = ['Pclass', 'Age', 'Fare', 'FamilySize', 'IsAlone']
categorical_features = ['Sex', 'Embarked', 'Title']

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onebot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ]
)

X_transformed = preprocessor.fit_transform(X)
print('Форма исходного X:', X.shape)
print('Форма после трансформации:', X_transformed.shape)
print('Есть ли теперь NaN в данных?:', np.isnan(X_transformed).any())

Форма исходного X: (891, 8)
Форма после трансформации: (891, 15)
Есть ли теперь NaN в данных?: False


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe_lr = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

cv_scores_lr = cross_val_score(pipe_lr, X_train, y_train, cv=cv, scoring='accuracy')

print('Фолды LR:', np.round(cv_scores_lr, 4))
print(f'Средняя точность (CV Accuracy): {cv_scores_lr.mean():.4f} +- {cv_scores_lr.std():.4f}')

Фолды LR: [0.8252 0.8042 0.8451 0.7958 0.838 ]
Средняя точность (CV Accuracy): 0.8216 +- 0.0190


In [17]:
pipe_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42))
])

cv_scores_rf = cross_val_score(pipe_rf, X_train, y_train, cv=cv, scoring='accuracy')

print('Фолды RF:', np.round(cv_scores_rf, 4))
print(f"Средняя точность RF: {cv_scores_rf.mean():.4f} +- {cv_scores_rf.std():.4f}")
print(f"Средняя точность LR: {cv_scores_lr.mean():.4f} ± {cv_scores_lr.std():.4f}")
diff = cv_scores_rf.mean() - cv_scores_lr.mean()
print(f"Разница (RF - LR): {diff:+.4f}")

Фолды RF: [0.8392 0.8042 0.8521 0.8028 0.8169]
Средняя точность RF: 0.8230 +- 0.0195
Средняя точность LR: 0.8216 ± 0.0190
Разница (RF - LR): +0.0014


In [19]:
param_grid = {
    'model__n_estimators': [50, 100, 150],
    'model__max_depth': [3,5,7,10],
    'model__min_samples_leaf': [1,2,4],
    'preprocessor__num__imputer__strategy': ['median', 'mean']
}

grid_rf = GridSearchCV(
    estimator=pipe_rf,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(X_train, y_train)

print('Лучшие параметры:')
for param, val in grid_rf.best_params_.items():
    print(f" {param}: {val}")

print(f'Лучшая модель на CV: {grid_rf.best_score_:.4f}')

Fitting 5 folds for each of 72 candidates, totalling 360 fits
Лучшие параметры:
 model__max_depth: 5
 model__min_samples_leaf: 1
 model__n_estimators: 50
 preprocessor__num__imputer__strategy: mean
Лучшая модель на CV: 0.8287


In [22]:
best_rf=grid_rf.best_estimator_

pipe_lr.fit(X_train, y_train)

y_pred_rf = best_rf.predict(X_test)
y_prob_rf = best_rf.predict_proba(X_test)[:,1]

y_pred_lr = pipe_lr.predict(X_test)
y_prob_lr = pipe_lr.predict_proba(X_test)[:, 1]

results = pd.DataFrame({
    "Метрика":["Accuracy", "F1-Score", "ROC-AUC"],
    "Logistic Regression": [
        accuracy_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_lr),
        roc_auc_score(y_test, y_prob_lr)
    ],
    'Tuned Random Forest': [
        accuracy_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_rf),
        roc_auc_score(y_test, y_prob_rf)
    ]
})

print('РЕЗУЛЬТАТЫ НА ОТЛОЖЕННОМ ТЕСТЕ (X_test):')
print(results.round(4).to_string(index=False))
        
    

РЕЗУЛЬТАТЫ НА ОТЛОЖЕННОМ ТЕСТЕ (X_test):
 Метрика  Logistic Regression  Tuned Random Forest
Accuracy               0.8380               0.8212
F1-Score               0.7820               0.7538
 ROC-AUC               0.8773               0.8572


In [23]:
seeds = [7, 13, 21, 42, 55, 77, 99, 123, 256, 777]
lr_test_scores = []
rf_test_scores = []

for s in seeds:
    # 1. Новое случайное разбиение
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=s, stratify=y)
    
    # 2. Обучаем LR
    pipe_lr.fit(X_tr, y_tr)
    lr_test_scores.append(accuracy_score(y_te, pipe_lr.predict(X_te)))
    
    # 3. Обучаем RF (с нашими лучшими параметрами)
    best_rf.fit(X_tr, y_tr)
    rf_test_scores.append(accuracy_score(y_te, best_rf.predict(X_te)))

print(f"LR на 10 seed: {np.mean(lr_test_scores):.4f} ± {np.std(lr_test_scores):.4f}")
print(f"RF на 10 seed: {np.mean(rf_test_scores):.4f} ± {np.std(rf_test_scores):.4f}")
print(f"Разница средних: {np.mean(rf_test_scores) - np.mean(lr_test_scores):+.4f}")

LR на 10 seed: 0.8307 ± 0.0166
RF на 10 seed: 0.8330 ± 0.0157
Разница средних: +0.0022
